<a href="https://colab.research.google.com/github/Jhonas05/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane (Week 5)

**Lane:** Refresh / Content Opportunity Scoring  
**Decision:** Which eligible content pages should a human editor inspect first?  
**Source:** FlyRank-approved pseudonymized starter CSV, `data/raw/content_refresh_anonymized.csv`  
**Comparison:** The **same Week-4 fixed-weight baseline**, measured alongside two simple learners **on the same client-held-out pages and identical descending-label proxy**.

**Read before running:** `skills/README.md`, `skills/training-honest-models/SKILL.md`, `skills/flyrank/flyrank-data/SKILL.md`, `docs/data-dictionary.md`, and `DATA_USE.md`. This notebook does **not** require a Hugging Face token or BigQuery. It uses only the already-approved starter dataset, which is **not** re-exported or printed as a full table.

**Critical scope:** The outcome is the *same-snapshot* `trend_direction == "down"` indicator: a directional **current-window proxy**, **not** future decline, editorial necessity, or evidence that refreshing content causes an improvement. Some 90-day activity features overlap the comparison period; that is acceptable **only for this descriptive snapshot-ranking exercise**, not for claiming prospective forecasting. A future period-specific task needs earlier features and later labels, as outlined in ML-04.

## 1. Method choice and why

My primary task is **ranking pages for a fixed editorial review budget**, not automatically changing content. Week 4's human-readable rule combines **days since last update (55%)** and **90-day impression visibility (45%)**. For ML-08, I compare:

- **Logistic Regression:** interpretable first learner; its predicted class probability provides a continuous queue score.
- **Random Forest (restricted depth):** a modest nonlinear comparator that may capture interactions between freshness, visibility, position, content characteristics, and click-through rate.
- **Week-4 fixed heuristic:** kept unchanged as the real baseline, not replaced by an alternative fitted rule.

The primary metric is **Precision@50**, reflecting the first 50 pages an editor would review. **P@10, P@20, ROC-AUC and average precision** are secondary checks. The preliminary target from Week 2 is an **absolute +0.05 in Precision@50** over the heuristic on held-out clients; it is a goal, *not* a guaranteed result. Complexity earns its keep only if the measured improvement and error pattern justify it.

Inputs: `days_since_last_update`, `log1p(impressions_90d)`, `content_age_days`, `ctr` (percent units), `avg_position`, `word_count`, and `content_type`. This small set adds understandable context beyond the two-input rule. Null word counts get training-only median imputation; zero `avg_position` means **missing position**, not a top rank. `content_type` missing values are imputed and one-hot encoded. The numeric model input scales use training-only statistics.

**Never use as model features:** IDs; `trend_direction`, `trend_pct`, `impressions_last_30d`, `impressions_prev_30d`, or their derived target; product flags; `provider_used` or `model_used`. While `impressions_90d` and `ctr` are safe from the *direct label-copy* problem, they overlap this same 90-day observational snapshot. Hence **no time-forward claim** is made.

In [12]:
from pathlib import Path
import subprocess
import sys
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display

SEED = 42
DATA_REL = Path("data/raw/content_refresh_anonymized.csv")

def find_repo_root():
    for folder in (Path.cwd(), *Path.cwd().parents):
        if (folder / DATA_REL).is_file():
            return folder
    if "google.colab" in sys.modules:
        dest = Path("/content/flyrank-ml-internship")
        if not (dest / DATA_REL).exists():
            if dest.exists():
                raise FileNotFoundError("An existing Colab folder lacks the approved CSV; inspect it before continuing.")
            subprocess.run(["git", "clone", "--depth", "1",
                            "https://github.com/Jhonas05/flyrank-ml-internship.git", str(dest)], check=True)
        return dest
    raise FileNotFoundError("Run in your repository clone, or Google Colab with network access.")

repo_root = find_repo_root()
df = pd.read_csv(repo_root / DATA_REL)
required = {"content_id", "client_id", "content_type", "impressions_90d",
            "days_since_last_update", "content_age_days", "ctr", "avg_position",
            "word_count", "trend_direction"}
assert required <= set(df), f"Missing expected starter fields: {sorted(required - set(df))}"
assert df["content_id"].is_unique, "Expected one unique content item per starter row"
lane = df.loc[(df["impressions_90d"] >= 100) &
              (df["content_age_days"] >= 90)].copy().reset_index(drop=True)
lane["declining_proxy"] = (lane["trend_direction"].astype(str).str.lower() == "down").astype(int)
assert len(lane) >= 100 and lane["client_id"].nunique() >= 8
print(f"Starter rows: {len(df):,}; eligible pages: {len(lane):,}; pseudonymized clients: {lane['client_id'].nunique()}")
print(f"Current-window declining proxy prevalence, eligible slice: {lane['declining_proxy'].mean():.1%}")
print(f"Versions: pandas {pd.__version__}, numpy {np.__version__}, scikit-learn {sklearn.__version__}")

Starter rows: 30,000; eligible pages: 22,006; pseudonymized clients: 30
Current-window declining proxy prevalence, eligible slice: 59.8%
Versions: pandas 2.2.3, numpy 2.1.3, scikit-learn 1.6.1


## 2. Split design

Use **GroupShuffleSplit(test_size=0.25, random_state=42)** on `client_id`. All content from any held-out client goes to **one side only**, preventing client-specific page records from appearing in both training and testing. IDs are never learned as predictors. The **baseline is not refit**; it is simply measured on the identical holdout used for the models.

This is **one** client-group holdout, not repeated cross-validation or a temporal holdout. Results may change with a different group split, and pages from the same client are not independent. Because there is only one trailing-90-day snapshot, **time-forward validation is not possible with this file**. ML-09 should test more groups/windows and temporal availability where data allow.

In [13]:
from sklearn.model_selection import GroupShuffleSplit

X = lane[["days_since_last_update", "content_age_days", "impressions_90d",
          "ctr", "avg_position", "word_count", "content_type"]].copy()
X["log_impressions_90d"] = np.log1p(
    pd.to_numeric(X.pop("impressions_90d"), errors="coerce").clip(lower=0))
X["avg_position"] = pd.to_numeric(X["avg_position"], errors="coerce").replace(0, np.nan)

NUMERIC = ["days_since_last_update", "content_age_days", "log_impressions_90d",
           "ctr", "avg_position", "word_count"]
CATEGORICAL = ["content_type"]
X[NUMERIC] = X[NUMERIC].apply(pd.to_numeric, errors="coerce")
X[CATEGORICAL] = X[CATEGORICAL].astype("object")
y = lane["declining_proxy"].astype(int)
groups = lane["client_id"]
BLOCKED = {"content_id", "client_id", "trend_direction", "trend_pct",
           "impressions_last_30d", "impressions_prev_30d", "declining_proxy",
           "is_declining_label", "provider_used", "model_used"}
assert not (BLOCKED & set(X.columns)), "A target-derived or prohibited field entered the predictors"
assert set(X.columns) == set(NUMERIC + CATEGORICAL)

split = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
train_idx, test_idx = next(split.split(X, y, groups))
assert set(groups.iloc[train_idx]).isdisjoint(set(groups.iloc[test_idx]))
assert len(test_idx) >= 50
assert y.iloc[train_idx].nunique() == 2 and y.iloc[test_idx].nunique() == 2
print(f"Training: {len(train_idx):,} pages / {groups.iloc[train_idx].nunique()} clients")
print(f"Holdout:  {len(test_idx):,} pages / {groups.iloc[test_idx].nunique()} clients")
print(f"Holdout proxy base rate: {y.iloc[test_idx].mean():.1%}")
print("No shared client groups; no pseudonymous IDs used as predictors.")

Training: 17,396 pages / 22 clients
Holdout:  4,610 pages / 8 clients
Holdout proxy base rate: 57.7%
No shared client groups; no pseudonymous IDs used as predictors.


## 3. Train + compare vs my baseline

**Same candidate slice:** at least 100 90-day impressions and 90 days of content age (Week-4 eligibility). **Same label:** `trend_direction == "down"` for all methods. **Same test clients/pages:** the indices above. **Same metrics:** evaluated once below.

The Week-4 baseline is reproduced exactly, including its dataset-wide visibility normalizer and rounding to four decimals:

`100 × (0.55 × min(days_since_last_update/180, 1) + 0.45 × log1p(impressions_90d)/log1p(max eligible impressions_90d))`

Its denominator is the visible aggregate across the **entire eligible snapshot**, as in Week 4. It does **not** depend on labels. That retains exact rule parity, although it is snapshot-specific and **not** a deployable pre-period normalizer for future forecasting.

Both learners use identical predictors and train-client-only preprocessing. Hyperparameters are fixed up front, **not tuned on the holdout**. A more complex approach is not automatically better. The **one final comparison table** is printed at the very end of this notebook.

In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

def preprocessing():
    # A fresh transformer for each trained model; fit learns only training-client statistics.
    return ColumnTransformer([
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median", add_indicator=True)),
            ("scale", StandardScaler()),
        ]), NUMERIC),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("encode", OneHotEncoder(handle_unknown="ignore")),
        ]), CATEGORICAL),
    ])

models = {
    "Logistic Regression": Pipeline([
        ("preprocess", preprocessing()),
        ("model", LogisticRegression(max_iter=1500, random_state=SEED)),
    ]),
    "Random Forest": Pipeline([
        ("preprocess", preprocessing()),
        ("model", RandomForestClassifier(n_estimators=120, max_depth=8,
                                          min_samples_leaf=12, random_state=SEED,
                                          n_jobs=-1)),
    ]),
}

# EXACT Week-4 fixed heuristic, recomputed on same eligible slice.
age = pd.to_numeric(lane["days_since_last_update"], errors="coerce").fillna(0).clip(lower=0)
visibility = pd.to_numeric(lane["impressions_90d"], errors="coerce").fillna(0).clip(lower=0)
max_visible = float(visibility.max())
assert max_visible >= 100
baseline_all = (100 * (0.55 * (age / 180).clip(upper=1) +
                       0.45 * (np.log1p(visibility) / np.log1p(max_visible)))).round(4)

scores_by_method = {"Week-4 fixed heuristic": baseline_all.iloc[test_idx].to_numpy(dtype=float)}
for label, model in models.items():
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    scores_by_method[label] = model.predict_proba(X.iloc[test_idx])[:, 1]
    assert np.isfinite(scores_by_method[label]).all()
    print(f"Trained {label}; calculated {len(test_idx):,} untouched holdout ranking scores")

def precision_at_k(y_true, scores, k):
    """Fraction of held-out pages with observed declining proxy among highest k scores."""
    values = np.asarray(y_true, dtype=int)
    scores = np.asarray(scores, dtype=float)
    assert len(values) == len(scores) and len(scores) >= k
    top = np.argsort(-scores, kind="stable")[:k]
    return float(values[top].mean())

y_test = y.iloc[test_idx].to_numpy(dtype=int)
base_rate = float(y_test.mean())
rows = []
for label, scores in scores_by_method.items():
    rows.append({
        "Method": label,
        "Precision@10": precision_at_k(y_test, scores, 10),
        "Precision@20": precision_at_k(y_test, scores, 20),
        "Precision@50": precision_at_k(y_test, scores, 50),
        "ROC-AUC": float(roc_auc_score(y_test, scores)),
        "Avg. precision": float(average_precision_score(y_test, scores)),
        "Holdout base rate": base_rate,
    })
comparison = pd.DataFrame(rows).set_index("Method")
assert len(comparison) == 3 and comparison.notna().all().all()
print("Computed held-out metrics for the Week-4 rule and both models using identical labels/pages.")

Trained Logistic Regression; calculated 4,610 untouched holdout ranking scores
Trained Random Forest; calculated 4,610 untouched holdout ranking scores
Computed held-out metrics for the Week-4 rule and both models using identical labels/pages.


## 4. Errors and interpretation

A high-ranking **non-declining** page is a costly *false priority* for an editor whose goal is to inspect currently declining content. A declining page ranked below the first 50 is a *missed opportunity within this proxy*. Neither category proves the content itself is good/bad or that editing it has a causal benefit.

Inspect **three concrete, de-identified wrong cases** using only non-identifying content attributes; look at the share of top-50 errors and the **three highest permutation importances**, measured by ROC-AUC decrease when each input is shuffled on the holdout. Importance represents *model reliance in this sample*, **not causation**. Compare subgroups only when group size is sufficient.

In [15]:
from sklearn.inspection import permutation_importance

# Logistic Regression is the prespecified, interpretable primary model.
primary_name = "Logistic Regression"
primary_scores = scores_by_method[primary_name]
ranked = np.argsort(-primary_scores, kind="stable")
rank_of_item = np.empty(len(y_test), dtype=int)
rank_of_item[ranked] = np.arange(1, len(y_test) + 1)

false_priorities = [i for i in ranked[:50] if y_test[i] == 0]
missed_proxies = [i for i in ranked[50:] if y_test[i] == 1]
print(f"{primary_name} top-50 non-declining proxy cases (false priorities): {len(false_priorities)}/50")
print(f"Declining-proxy pages not in top-50: {len(missed_proxies):,} (ranking capacity is fixed at 50)")

# Demonstrate concrete errors, WITHOUT printing client/content identifiers.
picks = [(i, "False priority in top-50") for i in false_priorities[:2]]
picks += [(i, "Declining proxy, below top-50") for i in missed_proxies[-max(0, 3 - len(picks)):]]
if len(picks) < 3:
    picks += [(i, "Declining proxy, below top-50") for i in missed_proxies[:3-len(picks)]]
case_rows = []
for n, (local_row, kind) in enumerate(picks[:3], 1):
    original = lane.iloc[test_idx[local_row]]
    why_hard = (
        "Signals put it inside the limited review budget, but its current trend does not show decline."
        if local_row in false_priorities else
        "The page has an observed declining trend, but many other pages rank ahead within the 50-page budget."
    )
    case_rows.append({
        "Case": f"{n} ({kind})",
        "Model rank": int(rank_of_item[local_row]),
        "Declining proxy": int(y_test[local_row]),
        "Model score": round(float(primary_scores[local_row]), 3),
        "90-day impressions": int(original["impressions_90d"]),
        "Days since update": int(original["days_since_last_update"]),
        "Content type": str(original["content_type"]),
        "Why this is difficult": why_hard,
    })
if case_rows:
    print("Three illustrative wrong priorities / misses (no IDs, URLs or client names):")
    display(pd.DataFrame(case_rows))
print("Interpretation: high visibility/freshness signals can rank a page highly without observed decline;")
print("similarly, true proxy-declining pages can rank low. Human reviewers must verify the actual editorial need.")

importance = permutation_importance(models[primary_name], X.iloc[test_idx], y_test,
                                    scoring="roc_auc", n_repeats=3,
                                    random_state=SEED, n_jobs=1)
importance_table = (pd.DataFrame({"Input feature": X.columns,
                                  "Mean AUC reduction": importance.importances_mean,
                                  "Std. dev.": importance.importances_std})
                    .sort_values("Mean AUC reduction", ascending=False)
                    .reset_index(drop=True))
print("Primary-model feature reliance (permutation importance on held-out clients):")
display(importance_table.round(4))
print("Top three inputs:", ", ".join(importance_table["Input feature"].head(3)))
print("Note: correlated inputs can split importance; negative/near-zero values need caution.")

# Aggregate-only content-type check; not group/client-identifying output.
summary = pd.DataFrame({
    "content_type": lane.iloc[test_idx]["content_type"].fillna("unknown").to_numpy(),
    "observed_proxy": y_test,
    "primary_score": primary_scores,
})
by_type = summary.groupby("content_type", dropna=False).agg(
    n=("observed_proxy", "size"),
    proxy_rate=("observed_proxy", "mean"),
    mean_model_score=("primary_score", "mean"),
)
print("Holdout content-type aggregates (only groups with n >= 30):")
display(by_type.loc[by_type["n"] >= 30].round(3))

Logistic Regression top-50 non-declining proxy cases (false priorities): 4/50
Declining-proxy pages not in top-50: 2,612 (ranking capacity is fixed at 50)
Three illustrative wrong priorities / misses (no IDs, URLs or client names):


,Case,Model rank,Declining proxy,Model score,90-day impressions,Days since update,Content type,Why this is difficult
0,1 (False priority in top-50),2,0,0.908,103,304,keyword article,Signals put it inside the limited review budge...
1,2 (False priority in top-50),14,0,0.861,1266,106,keyword article,Signals put it inside the limited review budge...
2,"3 (Declining proxy, below top-50)",4610,1,0.000,102,20,feedly article,"The page has an observed declining trend, but ..."


Interpretation: high visibility/freshness signals can rank a page highly without observed decline;
similarly, true proxy-declining pages can rank low. Human reviewers must verify the actual editorial need.
Primary-model feature reliance (permutation importance on held-out clients):


,Input feature,Mean AUC reduction,Std. dev.
0,ctr,0.0467,0.0029
1,content_age_days,0.0306,0.0027
2,log_impressions_90d,0.0303,0.0024
3,avg_position,0.0125,0.0019
4,days_since_last_update,0.0041,0.0014
5,word_count,-0.0038,0.0060
6,content_type,-0.0047,0.0005


Top three inputs: ctr, content_age_days, log_impressions_90d
Note: correlated inputs can split importance; negative/near-zero values need caution.
Holdout content-type aggregates (only groups with n >= 30):


,n,proxy_rate,mean_model_score
content_type,,,
comparison article,366,0.650,0.706
feedly article,232,0.703,0.482
keyword article,4012,0.563,0.565


### ML-08 — Results Interpretation

In the client-holdout evaluation, Logistic Regression achieved the highest Precision@50 at 0.920, compared with 0.420 for the fixed-rule baseline and 0.640 for Random Forest.

Logistic Regression also achieved the highest Precision@20 (0.850) and ROC-AUC (0.618). The observed decline prevalence in the test set was 57.7%.

These results suggest that Logistic Regression may provide more effective top-ranked content review recommendations for this evaluated dataset. However, the relatively modest ROC-AUC, single client-holdout split, and absence of independent future-month validation limit conclusions about generalization.

Further validation is necessary before recommending the model for operational deployment.

In [16]:
assert set(groups.iloc[train_idx]).isdisjoint(set(groups.iloc[test_idx]))
assert not (BLOCKED & set(X.columns))
assert np.isfinite(comparison.to_numpy()).all()
assert (comparison.select_dtypes(include="number") >= 0).all().all()
assert (comparison.select_dtypes(include="number") <= 1).all().all()

print("FINAL — Week-4 baseline vs ML-08 learners (held-out clients, identical test pages)")
display(comparison.round(3))
base_p50 = float(comparison.loc["Week-4 fixed heuristic", "Precision@50"])
for method in ("Logistic Regression", "Random Forest"):
    gain = float(comparison.loc[method, "Precision@50"] - base_p50)
    print(f"{method}: P@50 difference vs fixed baseline = {gain:+.3f} ({gain*100:+.1f} percentage points)")

winner = comparison["Precision@50"].idxmax()
print(f"Highest measured holdout P@50: {winner}. This is a comparison, NOT a deployment selection.")
print(f"Held-out proxy base rate = {base_rate:.1%}; test clients = {groups.iloc[test_idx].nunique()}; test pages = {len(test_idx):,}.")
print("Self-check: review all unchecked human validation items above before committing.")
print("Scope: directional editorial decision-support on one trailing-90-day snapshot; not a future forecast or causal test.")

FINAL — Week-4 baseline vs ML-08 learners (held-out clients, identical test pages)


,Precision@10,Precision@20,Precision@50,ROC-AUC,Avg. precision,Holdout base rate
Method,,,,,,
Week-4 fixed heuristic,0.9,0.65,0.42,0.413,0.522,0.577
Logistic Regression,0.9,0.85,0.92,0.618,0.692,0.577
Random Forest,0.6,0.50,0.64,0.609,0.666,0.577


Logistic Regression: P@50 difference vs fixed baseline = +0.500 (+50.0 percentage points)
Random Forest: P@50 difference vs fixed baseline = +0.220 (+22.0 percentage points)
Highest measured holdout P@50: Logistic Regression. This is a comparison, NOT a deployment selection.
Held-out proxy base rate = 57.7%; test clients = 8; test pages = 4,610.
Self-check: review all unchecked human validation items above before committing.
Scope: directional editorial decision-support on one trailing-90-day snapshot; not a future forecast or causal test.
